# Task 3: Combining datasets to answer RQs


**Question:  How popular were the top most recent Kdramas around the world in the month of September 2026?**

## 1. Assembling Top 10 Shows with Highest Ratings

Goal: produce a list of article names.

First, I looked up the top shows with the highest ratings manually and assembled a list:

1. Teach you a lesson 9.0 
2. Four Hands, Two Sonatas 8.7
3. Bloodhounds Season 2 8.7
4. Agent Kim Reactivated 8.6
5. Yumi's Cells Seaons 3 8.6
6. A Shop for Killers Season 2 8.5
7. The Wonderfools 8.5 
8. The legend of kitchen soldier 8.5
9. we are all trying here 8,5
10. A Bona Fide Killer 8.4

Next, we need a list of Wikipedia names. I also did this manually by looking at the unique URL of each title.

In [1]:
names = ['Teach_You_a_Lesson', 'Four_Hands,_Two_Sonatas', 'Bloodhounds_(TV_series)', 
         'Agent_Kim_Reactivated', 'Yumi%27s_Cells', 'A_Shop_for_Killers',
         'The_Wonderfools', 'The_Legend_of_Kitchen_Soldier','We_Are_All_Trying_Here',
         'A_Bona_Fide_Killer']

## 2. Retrieve Wiki Daily Pageviews (one day first)

The idea is to get the pageviews tsv of each day in September. For each day, search the df with the query (10 in total, 1 for each drama).

Let's just start with the example from the Wednesday notebook.

In [11]:
import pandas as pd
import requests, io

url = "https://analytics.wikimedia.org/published/datasets/country_project_page/2026-09-29.tsv"

headers = {'User-Agent': 'MyDataAnalysisScript/1.0 (contact: wendywellesley@gmail.com)'}
response = requests.get(url, headers=headers)


In [12]:
df = pd.read_csv(io.StringIO(response.text), sep='\t', header=None)
df.head()

,0,1,2,3,4,5,6
0,Algeria,DZ,ar.wikipedia,3947,Ø°ÙØ§Ø¡_Ø§ØµØ·ÙØ§Ø¹Ù,Q11660,122
1,Argentina,AR,es.wikipedia,3094323,Juana_Viale,Q9016209,115
2,Argentina,AR,es.wikipedia,327209,JosÃ©_Mourinho,Q79983,114
3,Argentina,AR,es.wikipedia,4449701,Copa_Libertadores,Q184795,386
4,Argentina,AR,es.wikipedia,47340,Thomas_Sankara,Q202155,119


Let's rename these columns.

In [13]:
columns = ['country', 'country-code', 'wiki', 'page_id', 'article', 'qid', 'views']
df.columns = columns

In [14]:
df.head()

,country,country-code,wiki,page_id,article,qid,views
0,Algeria,DZ,ar.wikipedia,3947,Ø°ÙØ§Ø¡_Ø§ØµØ·ÙØ§Ø¹Ù,Q11660,122
1,Argentina,AR,es.wikipedia,3094323,Juana_Viale,Q9016209,115
2,Argentina,AR,es.wikipedia,327209,JosÃ©_Mourinho,Q79983,114
3,Argentina,AR,es.wikipedia,4449701,Copa_Libertadores,Q184795,386
4,Argentina,AR,es.wikipedia,47340,Thomas_Sankara,Q202155,119


In [15]:
df.shape

(293466, 7)

Try to filter this df for specific queries:

In [7]:
query = names[0]

filtered_df = df[
    df['article'].str.contains(query, case=False, na=False)
]

filtered_df.head()

,country,country-code,wiki,page_id,article,qid,views
5054,Singapore,SG,en.wikipedia,79560156,Teach_You_a_Lesson,Q137844128,111
8713,United States,US,en.wikipedia,79560156,Teach_You_a_Lesson,Q137844128,369
38494,India,IN,en.wikipedia,79560156,Teach_You_a_Lesson,Q137844128,244
75513,Indonesia,ID,en.wikipedia,79560156,Teach_You_a_Lesson,Q137844128,129
87615,Philippines,PH,en.wikipedia,79560156,Teach_You_a_Lesson,Q137844128,139


In [10]:
filtered_df.shape

(5, 7)

This means that on this day, only five countries looked this kdrama up on Wikipedia, and each country had a certain number of views.

## 3. Set up loop for URL updates to capture entire September

This is a simple but important step! We will automate the collection of all tsvs in September by making the URL a f-string.

In [29]:
sept_urls = []
for i in range(1,31):  # 30 days in september
    if i < 10:
        date = f"2026-09-0{i}"
    else:
        date = f"2026-09-{i}"
    url = f"https://analytics.wikimedia.org/published/datasets/country_project_page/{date}.tsv"
    sept_urls.append(url)

In [30]:
sept_urls[:5]

['https://analytics.wikimedia.org/published/datasets/country_project_page/2026-09-01.tsv',
 'https://analytics.wikimedia.org/published/datasets/country_project_page/2026-09-02.tsv',
 'https://analytics.wikimedia.org/published/datasets/country_project_page/2026-09-03.tsv',
 'https://analytics.wikimedia.org/published/datasets/country_project_page/2026-09-04.tsv',
 'https://analytics.wikimedia.org/published/datasets/country_project_page/2026-09-05.tsv']

## 4. 

Goal: columns = date, country, article, pageviews; each row is one country one article.

Pseudocode for the full block of code:

create: kdrama_df    
for every url:    
-> for every query:   
--> create a column for date that is just the date  
--> merge the smaller filtered_df with the larger kdrama_df

In [26]:
columns = ['country', 'country-code', 'wiki', 'page_id', 'article', 'qid', 'views']
full_df = pd.DataFrame()

for i in range(10, len(sept_urls)):
    date = f"2026-09-{i}"
    url = sept_urls[i]

    # get the daily tsv first
    headers = {'User-Agent': 'MyDataAnalysisScript/1.0 (contact: wendywellesley@gmail.com)'}
    response = requests.get(url, headers=headers)

    # read in as df
    day_df = pd.read_csv(io.StringIO(response.text), sep='\t', header=None)
    day_df.columns = columns

    # add a column for date
    day_df['date'] = date

    for j in range(1, len(names)): # each kdrama
        query = names[j]
        filtered_df = day_df[
            day_df['article'].str.contains(query, case=False, na=False)]
        full_df = pd.concat([full_df, filtered_df], axis=1)


                    0   1                 2         3  \
0           Argentina  AR      es.wikipedia  11526564   
1           Argentina  AR      es.wikipedia   3094323   
2           Argentina  AR      es.wikipedia    327209   
3           Argentina  AR      es.wikipedia   4449701   
4           Argentina  AR      es.wikipedia   5640452   
...               ...  ..               ...       ...   
304478  United States  US  simple.wikipedia    525641   
304479  United States  US      zh.wikipedia    124440   
304480  United States  US      zh.wikipedia       317   
304481  United States  US      zh.wikipedia     86555   
304482        Uruguay  UY      es.wikipedia  11552922   

                                         4           5    6  
0       La_invitaciÃ³n_(pelÃ­cula_de_2026)  Q134083296  429  
1                              Juana_Viale    Q9016209  181  
2                           JosÃ©_Mourinho      Q79983  153  
3            Copa_Libertadores_de_AmÃ©rica     Q184795  716  
4    

/var/folders/f6/71_23hqx2gv3g0x_h49kbbf80000gn/T/ipykernel_10388/2281861999.py:23: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  day_df['article'].str.contains(query, case=False, na=False)]


                    0   1                 2         3  \
0             Algeria  DZ      ar.wikipedia   9046254   
1           Argentina  AR      en.wikipedia  80387746   
2           Argentina  AR      es.wikipedia  11526564   
3           Argentina  AR      es.wikipedia   3094323   
4           Argentina  AR      es.wikipedia    327209   
...               ...  ..               ...       ...   
318185  United States  US      or.wikipedia      1594   
318186  United States  US     ru.wikivoyage      8876   
318187  United States  US  simple.wikipedia    525641   
318188  United States  US      zh.wikipedia     86555   
318189        Uruguay  UY      es.wikipedia  11552922   

                                                        4           5    6  
0                                   Ø¥ÙÙÙÙ_ÙØ®ØªÙÙ   Q64168758  156  
1                                         Inde_Navarrette  Q106932301  119  
2                      La_invitaciÃ³n_(pelÃ­cula_de_2026)  Q134083296  757  
3      

/var/folders/f6/71_23hqx2gv3g0x_h49kbbf80000gn/T/ipykernel_10388/2281861999.py:23: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  day_df['article'].str.contains(query, case=False, na=False)]


                    0   1              2         3  \
0           Argentina  AR   es.wikipedia  11526564   
1           Argentina  AR   es.wikipedia   2934940   
2           Argentina  AR   es.wikipedia   3094323   
3           Argentina  AR   es.wikipedia    327209   
4           Argentina  AR   es.wikipedia   4449701   
...               ...  ..            ...       ...   
340091  United States  US  ru.wikivoyage      8876   
340092  United States  US  ru.wikivoyage       999   
340093  United States  US   zh.wikipedia    124440   
340094  United States  US   zh.wikipedia     86555   
340095        Uruguay  UY   es.wikipedia    148834   

                                                        4           5    6  
0                      La_invitaciÃ³n_(pelÃ­cula_de_2026)  Q134083296  716  
1                                      VÃ­ctor_Sarasqueta    Q5671904  116  
2                                             Juana_Viale    Q9016209  464  
3                                          

...

/var/folders/f6/71_23hqx2gv3g0x_h49kbbf80000gn/T/ipykernel_10388/2281861999.py:23: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  day_df['article'].str.contains(query, case=False, na=False)]


                    0   1             2         3                        4  \
0             Algeria  DZ  ar.wikipedia      3947  Ø°ÙØ§Ø¡_Ø§ØµØ·ÙØ§Ø¹Ù   
1           Argentina  AR  es.wikipedia   3094323              Juana_Viale   
2           Argentina  AR  es.wikipedia    327209           JosÃ©_Mourinho   
3           Argentina  AR  es.wikipedia   4449701        Copa_Libertadores   
4           Argentina  AR  es.wikipedia     47340           Thomas_Sankara   
...               ...  ..           ...       ...                      ...   
293461  United States  US  zh.wikipedia    124440                èä¸æ³¢   
293462  United States  US  zh.wikipedia       317             äººå·¥æºè½   
293463  United States  US  zh.wikipedia   6331946                é§é¾å¨   
293464  United States  US  zh.wikipedia     86555                   æ±é   
293465        Uruguay  UY  es.wikipedia  11552922    Copa_AUF_Uruguay_2026   

                 5    6  
0           Q11660  122  
1         Q

/var/folders/f6/71_23hqx2gv3g0x_h49kbbf80000gn/T/ipykernel_10388/2281861999.py:23: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  day_df['article'].str.contains(query, case=False, na=False)]


                    0   1             2         3  \
0             Algeria  DZ  ar.wikipedia      3947   
1           Argentina  AR  es.wikipedia  11609897   
2           Argentina  AR  es.wikipedia   3094323   
3           Argentina  AR  es.wikipedia    327209   
4           Argentina  AR  es.wikipedia   4449701   
...               ...  ..           ...       ...   
285746  United States  US  zh.wikipedia   5799092   
285747  United States  US  zh.wikipedia   6331946   
285748  United States  US  zh.wikipedia     86555   
285749  United States  US  zh.wikipedia   9622536   
285750        Uruguay  UY  es.wikipedia  11552922   

                                                  4           5     6  
0                           Ø°ÙØ§Ø¡_Ø§ØµØ·ÙØ§Ø¹Ù      Q11660    95  
1                                 Jhon_Jairo_Torres         NaN   182  
2                                       Juana_Viale    Q9016209  4958  
3                                    JosÃ©_Mourinho      Q79983   110  
4  

/var/folders/f6/71_23hqx2gv3g0x_h49kbbf80000gn/T/ipykernel_10388/2281861999.py:23: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  day_df['article'].str.contains(query, case=False, na=False)]


In [28]:
full_df.tail()

,country,country-code,wiki,page_id,article,qid,views,date,country,country-code,...,views,date,country,country-code,wiki,page_id,article,qid,views,date
26086,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
132138,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
76924,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Philippines,PH,en.wikipedia,83488921.0,A_Bona_Fide_Killer,Q139411314,131.0,2026-09-29
169926,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,India,IN,en.wikipedia,83488921.0,A_Bona_Fide_Killer,Q139411314,94.0,2026-09-29
232718,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,United States,US,en.wikipedia,83488921.0,A_Bona_Fide_Killer,Q139411314,250.0,2026-09-29
